# Walmart App Onboarding Tutorial

This tutorial demonstrates how to:
1. Create an AndroidWorld environment
2. Install an app on the emulator
3. Design onboarding rules using screenshots and tools

We'll use the Walmart app as an example, capturing screenshots at each step to determine which buttons to click.


## Setup

First, let's set up the environment and import necessary modules.


In [ ]:
import os
import sys
import time
import shutil
import tempfile

# CRITICAL: Patch tempfile.gettempdir() before importing android_world
# This is needed because in Jupyter, tempfile is already imported and gettempdir() is cached
_AW_TMPDIR = os.path.join(os.getcwd(), ".aw_tmp")
os.makedirs(_AW_TMPDIR, exist_ok=True)

# Patch tempfile.gettempdir to use our custom directory
_original_gettempdir = tempfile.gettempdir
tempfile.gettempdir = lambda: _AW_TMPDIR

# Also set environment variables for good measure
os.environ['TMPDIR'] = _AW_TMPDIR
os.environ['ANDROID_HOME'] = os.environ.get('ANDROID_HOME', '/shared/ken/.android')
os.environ['ANDROID_SDK_ROOT'] = os.environ.get('ANDROID_SDK_ROOT', '/shared/ken/.android')
os.environ['GRPC_VERBOSITY'] = 'ERROR'
os.environ['GRPC_TRACE'] = 'none'

from android_world.env import interface
from android_world.env import env_launcher
from android_world.env.setup_device import setup as setup_module
from android_world.env.setup_device.bmoca_apps import WalmartApp, WikipediaApp, InstagramApp
from android_world.env import tools
from android_world.utils import screenshot_utils

print(f"✓ Temp directory configured: {_AW_TMPDIR}")
print(f"✓ tempfile.gettempdir() returns: {tempfile.gettempdir()}")
print(f"✓ Directory exists and is writable: {os.access(_AW_TMPDIR, os.W_OK)}")


✓ Temp directory configured: /home/ligu/projects/android_world_plus/.aw_tmp
✓ tempfile.gettempdir() returns: /home/ligu/projects/android_world_plus/.aw_tmp
✓ Directory exists and is writable: True


## Step 1: Create Environment

We connect to an existing emulator using `env_launcher._get_env()`. This creates an `AsyncEnv` interface that allows us to interact with the Android device.


In [2]:

print("Creating environment (connecting to emulator on console_port=5554)...")
adb_path = shutil.which("adb") or '~/Android/Sdk/platform-tools/adb'

# Connect to existing emulator
env: interface.AsyncEnv = env_launcher._get_env(
    console_port=5554,
    adb_path=adb_path,
    grpc_port=8554,
)

print("✓ Environment created successfully")


Creating environment (connecting to emulator on console_port=5554)...


E0000 00:00:1762200940.890013  605866 trace.cc:89] Unknown tracer: none


✓ Environment created successfully


## Step 2: Install App (Walmart)

If the Walmart app is not already installed, we can use `setup_module.maybe_install_app()` to install it. This function checks if the app is already installed and only installs if needed.

Before making a demo of "Installation & Setup/Onboarding", please delete the app first. Using the command `adb shell pm uninstall com.walmart.android` / `adb shell pm uninstall org.wikipedia`


In [14]:
try:
    print("Attempting to install Walmart app if needed...")
    setup_module.maybe_install_app(WalmartApp, env)
    print("✓ WalmartApp installation completed")
    setup_module.maybe_install_app(WikipediaApp, env)
    print("✓ WikipediaApp installation completed")
except Exception as install_err:
    # App may already be installed
    print(f"App install step: {install_err}")

from android_world.env import adb_utils
print("Running WikipediaApp.setup() - the automated onboarding rule...")
WikipediaApp.setup(env)
print("\n✓ Onboarding completed!")
time.sleep(3.0)
adb_utils.close_app(WalmartApp.app_name, env.controller)
print("✓ Closed app")

Attempting to install Walmart app if needed...
✓ WalmartApp installation completed
✓ WikipediaApp installation completed
Running WikipediaApp.setup() - the automated onboarding rule...
[WikipediaApp] Starting setup for wikipedia...
[WikipediaApp] ✓ Cleared app data successfully
[WikipediaApp] Launching app...
[WikipediaApp] ✓ App launched successfully
[WikipediaApp] Created AndroidToolController
[WikipediaApp] Initial screen: .wikipedia_debug_screenshots/screenshot_20251103_204113_114908.png
[WikipediaApp] Looking for Skip button...
[WikipediaApp] Found Skip button at: (84, 2274)
[WikipediaApp] ✓ Clicked Skip button
[WikipediaApp] Final screen: .wikipedia_debug_screenshots/screenshot_20251103_204115_331183.png
[WikipediaApp] ✓ Onboarding flow completed successfully
[WikipediaApp] Closing app...
[WikipediaApp] ✓ Setup completed
[WikipediaApp] Note: Debug screenshots saved to .wikipedia_debug_screenshots/ if any issues occurred

✓ Onboarding completed!
✓ Closed app


## Step 3: Understanding the Onboarding process (Walmart)

Before we can automate onboarding, we need to understand what screens appear during the first launch. Let's manually walk through the process and capture screenshots at each step.


### Step 3.1: Design Onboarding Rule

Based on the screenshot and UI elements, we can now design our onboarding rule. Looking at the UI elements, we need to determine:

1. Which button/text should we click first?
2. What's the next step?
3. Are there any permission dialogs?

Let's create a `Controller` to interact with the UI:


In [29]:
controller = tools.AndroidToolController(env=env.controller)
print("✓ Created AndroidToolController")


✓ Created AndroidToolController


### Step 3.2: Execute Step-by-Step Onboarding

Now we'll manually step through each screen, deciding what to click next. This helps us design the automated onboarding rule.


In [32]:
from android_world.env import adb_utils

# Launch app
print("Launching Walmart app...")
adb_utils.launch_app(WalmartApp.app_name, env.controller)
time.sleep(3.0)  # Wait for splash screen
print("✓ App launched")

# STEP 1: Handle initial login/guest mode
print("\n=== STEP 1/4: Click 'Continue as guest' ===")
try:
    controller.click_element("Continue as guest")
    time.sleep(1.0)
    print("✓ Clicked 'Continue as guest' button")
except ValueError as e:
    print(f"⚠ Button not found: {e}")

# Inspect UI after step 1
state = env.get_state()
print("\nUI elements after step 1:")
print("="*80)
for i, elem in enumerate(state.ui_elements[:20]):
    print(f"{i}: text='{elem.text}' | desc='{elem.content_description}' | "
          f"clickable={elem.is_clickable}")


# STEP 2: Handle "Maybe later" option
print("\n=== STEP 2/4: Click 'Maybe later' ===")
try:
    controller.click_element("Maybe later")
    time.sleep(1.3)
    print("✓ Clicked 'Maybe later' button")
except ValueError as e:
    print(f"⚠ Button not found: {e}")


# STEP 3: Handle location permission - Share precise location
print("\n=== STEP 3/4: Click 'Share precise location' ===")
try:
    controller.click_element("Share precise location")
    time.sleep(1.4)
    print("✓ Clicked 'Share precise location' button")
except ValueError as e:
    print(f"⚠ Button not found: {e}")

# STEP 4: Handle location permission - While using the app
print("\n=== STEP 4/4: Click 'While using the app' ===")
try:
    controller.click_element("While using the app")
    time.sleep(1.2)
    print("✓ Clicked 'While using the app' button")
except ValueError as e:
    print(f"⚠ Button not found: {e}")

# Close the app first
time.sleep(3.0)
adb_utils.close_app(WalmartApp.app_name, env.controller)
print("✓ Closed app")
print("\nNow we would implement the complete rule based on our analysis.")


Launching Walmart app...
✓ App launched

=== STEP 1/4: Click 'Continue as guest' ===
✓ Clicked 'Continue as guest' button

UI elements after step 1:
0: text='None' | desc='navigate back' | clickable=True
1: text='Get order and savings updates' | desc='None' | clickable=False
2: text='We’ll notify you about order status, special savings, store events and more.' | desc='None' | clickable=False
3: text='Get notifications' | desc='None' | clickable=True
4: text='Maybe later' | desc='None' | clickable=True
5: text='8:02' | desc='8:02 PM' | clickable=False
6: text='None' | desc='Android System notification: Serial console enabled' | clickable=False
7: text='None' | desc='Wifi signal full.' | clickable=False
8: text='None' | desc='Phone three bars.' | clickable=False
9: text='None' | desc='Battery charging, 100 percent.' | clickable=False

=== STEP 2/4: Click 'Maybe later' ===
✓ Clicked 'Maybe later' button

=== STEP 3/4: Click 'Share precise location' ===
✓ Clicked 'Share precise location' b

## Step 4: Using the Automated Rule

Once we've designed the rule by inspecting screenshots and UI elements, we can use the `WalmartApp.setup()` method which implements the complete automated onboarding flow.


In [3]:
print("Attempting to install Walmart app if needed...")
setup_module.maybe_install_app(WalmartApp, env)
print("✓ App installation completed")

print("Running WalmartApp.setup() - the automated onboarding rule...")
WalmartApp.setup(env)
print("\n✓ Onboarding completed!")


Attempting to install Walmart app if needed...
✓ App installation completed
Running WalmartApp.setup() - the automated onboarding rule...
[WalmartApp] Starting setup for walmart...
[WalmartApp] ✓ Cleared app data successfully
[WalmartApp] Launching app...
[WalmartApp] ✓ App launched successfully
[WalmartApp] Created AndroidToolController
[WalmartApp] Step 1/4: Click 'Continue as guest'...
[WalmartApp] ✓ Clicked 'Continue as guest' button
[WalmartApp] Step 2/4: Click 'Maybe later'...
[WalmartApp] ✓ Clicked 'Maybe later' button
[WalmartApp] Step 3/4: Click 'Share precise location'...
[WalmartApp] ✓ Clicked 'Share precise location' button
[WalmartApp] Step 4/4: Click 'While using the app'...
[WalmartApp] ✓ Clicked 'While using the app' button
[WalmartApp] ✓ Onboarding flow completed successfully
[WalmartApp] Closing app...
[WalmartApp] ✓ Setup completed

✓ Onboarding completed!


## Step 5: Verifying save & restore snapshot for Task Initialization


This section demonstrates how to verify the `_initialize_apps` function from `TaskEval` class. The `_initialize_apps` function is responsible for restoring app snapshots for all apps used in a task (except clipper app).

Let's verify this functionality works correctly for apps like Wikipedia.


### Creating and Saving a Walmart Snapshot

1. Running `WalmartApp.setup()` to get the app into a good state (after onboarding)
2. Saving a snapshot using `app_snapshot.save_snapshot()`

**⚠️ IMPORTANT:** The app must be launched and initialized (data directory created) before saving a snapshot. The `setup()` method closes the app, so we may need to launch it again briefly to ensure data exists.


In [10]:
# Step 1: Setup Walmart app to create a good state (after onboarding)
from android_world.env.setup_device.bmoca_apps import WalmartApp
from android_world.utils import app_snapshot

# Step 1.1: Re-launch app to ensure data directory exists
print("Step 1.1: Re-launching app to ensure data directory exists...")
adb_utils.launch_app("walmart", env.controller)
time.sleep(3.0)  # Give app time to initialize and create data directories
print("✓ App launched and initialized\n")

# Step 2: Save a snapshot of the current app state
print("Step 2: Saving snapshot of Walmart app state...")
try:
    app_snapshot.save_snapshot("walmart", env.controller)
    print("✓ Snapshot saved successfully\n")
except Exception as e:
    print(f"✗ Error saving snapshot: {e}\n")

# Step 3: Close the app
adb_utils.close_app("walmart", env.controller)
print("✓ App closed\n")

# Step 4: Now verify that we can restore it
print("Step 3: Verifying snapshot restore...")
try:
    app_snapshot.restore_snapshot("walmart", env.controller)
    print("✓ Snapshot restored successfully")
except RuntimeError as error:
    print(f"✗ Snapshot restore FAILED: {error}")
except Exception as e:
    print(f"✗ Unexpected error: {e}")


Step 1.1: Re-launching app to ensure data directory exists...
✓ App launched and initialized

Step 2: Saving snapshot of Walmart app state...
✓ Snapshot saved successfully

✓ App closed

Step 3: Verifying snapshot restore...
✓ Snapshot restored successfully


In [9]:
from android_world.utils import app_snapshot
from android_world.env.setup_device import setup as setup_module
from android_world.env.setup_device.bmoca_apps import WikipediaApp
from absl import logging
from android_world.env import adb_utils

def verify_app_initialization(app_names, env):
    """
    Verify the _initialize_apps function logic for given app names.
    
    This mimics the behavior of TaskEval._initialize_apps():
    - Checks if apps are installed
    - Attempts to restore snapshots (except for clipper app)
    - Handles errors gracefully
    
    Args:
        app_names: List or tuple of app names to verify
        env: AsyncEnv instance
    
    Returns:
        dict: Verification results for each app
    """
    results = {}
    
    print(f"Verifying app initialization for: {app_names}")
    print("=" * 80)
    
    for app_name in app_names:
        print(f"\n📱 Processing app: {app_name}")
        results[app_name] = {
            'installed': False,
            'snapshot_restored': False,
            'error': None
        }
        
        # Check if app is installed
        try:
            # Get package name
            activity = adb_utils.get_adb_activity(app_name)
            if activity:
                package_name = adb_utils.extract_package_name(activity)
                is_installed = setup_module.is_package_installed(package_name, env)
                results[app_name]['installed'] = is_installed
                
                if is_installed:
                    print(f"  ✓ App '{app_name}' is installed (package: {package_name})")
                else:
                    print(f"  ✗ App '{app_name}' is NOT installed (package: {package_name})")
            else:
                print(f"  ⚠ Could not find activity mapping for '{app_name}'")
                results[app_name]['error'] = "Activity mapping not found"
        except Exception as e:
            print(f"  ✗ Error checking installation: {e}")
            results[app_name]['error'] = str(e)
            continue
        
        # Don't need to restore snapshot for clipper app (as per TaskEval logic)
        if app_name and app_name != "clipper":
            try:
                print(f"  🔄 Attempting to restore snapshot for '{app_name}'...")
                app_snapshot.restore_snapshot(app_name, env.controller)
                results[app_name]['snapshot_restored'] = True
                print(f"  ✓ Snapshot restored successfully for '{app_name}'")
            except RuntimeError as error:
                # This is expected if no snapshot exists yet
                print(f"  ⚠ Snapshot restore skipped (no snapshot exists): {error}")
                results[app_name]['error'] = f"Snapshot not found: {error}"
            except Exception as e:
                print(f"  ✗ Error restoring snapshot: {e}")
                results[app_name]['error'] = str(e)
        else:
            print(f"  ⏭ Skipping snapshot restore for '{app_name}' (clipper app)")
    
    print("\n" + "=" * 80)
    print("📊 Verification Summary:")
    for app_name, result in results.items():
        status = "✓" if result['installed'] else "✗"
        snapshot_status = "✓" if result['snapshot_restored'] else "⚠"
        print(f"  {status} {app_name}: installed={result['installed']}, snapshot={snapshot_status}")
        if result['error']:
            print(f"    Error: {result['error']}")
    
    return results

# Test with Walmart app
print("Testing app initialization verification...")
results = verify_app_initialization(("walmart",), env)


Testing app initialization verification...
Verifying app initialization for: ('walmart',)

📱 Processing app: walmart
  ✓ App 'walmart' is installed (package: com.walmart.android)
  🔄 Attempting to restore snapshot for 'walmart'...
  ✓ Snapshot restored successfully for 'walmart'

📊 Verification Summary:
  ✓ walmart: installed=True, snapshot=✓


### Creating and Saving a Snapshot

Before we can restore a snapshot, we need to create one. Let's first setup Wikipedia app (which creates a good state), then save a snapshot that can be restored later.

**⚠️ IMPORTANT:** The `copy_dir` function silently ignores missing source directories. If `/data/data/org.wikipedia` doesn't exist, the save will appear successful but won't actually copy anything!


In [ ]:
# Debug helper to check snapshot paths and contents
from android_world.utils import app_snapshot
from android_world.env import adb_utils
from android_world.utils import file_utils
from android_env.proto import adb_pb2

def debug_snapshot_paths(app_name, env):
    """Debug snapshot and app data paths to understand save/restore issues."""
    from android_world.utils.app_snapshot import _app_data_path, _snapshot_path
    
    app_data = _app_data_path(app_name)
    snapshot = _snapshot_path(app_name)
    
    print(f"Debugging snapshot paths for: {app_name}")
    print("=" * 80)
    print(f"\n1. App Data Path: {app_data}")
    app_data_exists = file_utils.check_directory_exists(app_data, env)
    print(f"   Exists: {'✓ YES' if app_data_exists else '✗ NO'}")
    
    if app_data_exists:
        # Try to list contents
        try:
            result = adb_utils.issue_generic_request(
                ["shell", "ls", "-la", app_data], env
            )
            if result.status == adb_pb2.AdbResponse.Status.OK:
                contents = result.generic.output.decode('utf-8').strip()
                if contents:
                    lines = contents.split('\n')
                    print(f"   Contents preview ({len(lines)} items):")
                    for line in lines[:5]:  # First 5 lines
                        print(f"     {line}")
                else:
                    print(f"   ⚠ Directory exists but appears empty")
        except Exception as e:
            print(f"   ⚠ Could not list contents: {e}")
    else:
        print(f"   ⚠️ CRITICAL: App data directory does NOT exist!")
        print(f"   This means the app hasn't created its data directory yet.")
        print(f"   Solution: Launch and use the app first to create data!")
    
    print(f"\n2. Snapshot Path: {snapshot}")
    snapshot_exists = file_utils.check_directory_exists(snapshot, env)
    print(f"   Exists: {'✓ YES' if snapshot_exists else '✗ NO'}")
    
    if snapshot_exists:
        # Check if snapshot has content
        try:
            result = adb_utils.issue_generic_request(
                ["shell", "ls", snapshot], env
            )
            if result.status == adb_pb2.AdbResponse.Status.OK:
                contents = result.generic.output.decode('utf-8').strip()
                if contents:
                    items = [l for l in contents.split('\n') if l.strip()]
                    print(f"   Contains: {len(items)} items")
                    print(f"   Sample items: {items[:5]}")  # First 5 items
                else:
                    print(f"   ⚠️ CRITICAL: Snapshot directory exists but is EMPTY!")
                    print(f"   This explains why restore fails!")
        except Exception as e:
            print(f"   ⚠ Could not list contents: {e}")
    
    print("\n" + "=" * 80)
    
    # The problem: copy_dir silently returns if source doesn't exist!
    if not app_data_exists:
        print("⚠️ ROOT CAUSE IDENTIFIED:")
        print(f"   App data directory '{app_data}' does NOT exist!")
        print("\n   What happens in save_snapshot():")
        print("   1. save_snapshot() calls copy_dir(source, dest)")
        print("   2. copy_dir() checks: does source exist? → NO")
        print("   3. copy_dir() logs a warning and RETURNS SILENTLY")
        print("   4. NO exception is raised → save appears 'successful'")
        print("   5. But snapshot is empty/missing → restore fails!")
        print("\n   SOLUTION:")
        print("   - Launch the app and interact with it first")
        print("   - This creates the /data/data/org.wikipedia directory")
        print("   - THEN save the snapshot")
    
    return {
        'app_data_path': app_data,
        'app_data_exists': app_data_exists,
        'snapshot_path': snapshot,
        'snapshot_exists': snapshot_exists
    }

# Step 1: Setup Wikipedia app to create a good state
from android_world.env.setup_device.bmoca_apps import WikipediaApp
from android_world.env import adb_utils

print("Step 1: Setting up Wikipedia app...")
WikipediaApp.setup(env)
print("✓ Wikipedia app setup completed\n")

# IMPORTANT: After setup(), the app is closed. 
# We need to launch it again and let it create data before saving snapshot!
print("Step 1.1: Re-launching app to ensure data directory exists...")
adb_utils.launch_app("wikipedia", env.controller)
import time
time.sleep(2.0)  # Give app time to initialize and create data directories
print("✓ App launched and initialized\n")

# Step 1.5: Debug paths BEFORE saving
print("Step 1.5: Debugging paths BEFORE saving snapshot...")
# Note: env.controller is AndroidWorldController which wraps env_interface
# For compatibility, we use env.controller.env which is the underlying AndroidEnvInterface
try:
    debug_result = debug_snapshot_paths("wikipedia", env.controller.env)
except AttributeError:
    # If env.controller doesn't have .env, try direct access
    debug_result = debug_snapshot_paths("wikipedia", env.controller)

# Step 2: Save a snapshot of the current app state
print("\nStep 2: Saving snapshot of Wikipedia app state...")
try:
    app_snapshot.save_snapshot("wikipedia", env.controller)
    print("✓ Snapshot save function completed (but may have silently failed if source missing)\n")
except Exception as e:
    print(f"✗ Error saving snapshot: {e}\n")

# Step 2.5: Debug paths AFTER saving
print("Step 2.5: Debugging paths AFTER saving snapshot...")
try:
    debug_result_after = debug_snapshot_paths("wikipedia", env.controller.env)
except AttributeError:
    debug_result_after = debug_snapshot_paths("wikipedia", env.controller)

# Step 3: Now verify that we can restore it
print("\nStep 3: Verifying snapshot restore...")
try:
    app_snapshot.restore_snapshot("wikipedia", env.controller)
    print("✓ Snapshot restored successfully")
except RuntimeError as error:
    print(f"✗ Snapshot restore FAILED: {error}")
    print("\nThis usually means:")
    print("  1. Snapshot directory doesn't exist, OR")
    print("  2. Snapshot directory exists but is empty")
    print("\nCheck the debug output above to see what happened!")
except Exception as e:
    print(f"✗ Unexpected error: {e}")


### 🔍 Root Cause Analysis: Why "Snapshot Saved Successfully" But "Snapshot Not Found"

**The Problem:**

Looking at the code in `android_world/utils/file_utils.py`, the `copy_dir()` function has this behavior:

```python
def copy_dir(source_path, dest_path, env):
    if not check_directory_exists(source_path, env):
        logging.warn("Source directory %s does not exist, ignoring copy_dir.", source_path)
        return  # ⚠️ Returns silently without raising an error!
    
    # ... copy logic ...
```

**What happens:**

1. `save_snapshot()` calls `copy_dir(source="/data/data/org.wikipedia", dest="/data/data/android_world/snapshots/org.wikipedia")`
2. If the source directory doesn't exist, `copy_dir()` logs a warning and returns silently
3. No exception is raised → `save_snapshot()` appears "successful"
4. But nothing was copied → snapshot directory is empty/missing
5. `restore_snapshot()` checks if snapshot exists → fails with "Snapshot not found"

**Why the source directory might not exist:**

- App needs to be **launched and initialized** to create `/data/data/org.wikipedia`
- Just installing the app doesn't create the data directory
- `WikipediaApp.setup()` closes the app at the end, so data might not be fully initialized
- The app data directory is created on first launch when Android initializes the app

**Solution:**

1. Launch the app after setup (done in Step 1.1 above)
2. Wait a moment for the app to initialize its data directory
3. THEN save the snapshot

**Check the debug output above to see:**
- Does `/data/data/org.wikipedia` exist before saving? (Should be YES)
- Does snapshot directory exist after saving? (Should be YES with contents)


### Testing with Multiple Apps

Now let's test the verification function with multiple apps to see how it behaves:


In [ ]:
# Test with multiple apps
test_apps = ("wikipedia", "instagram", "walmart", "clipper")

print("Testing app initialization verification with multiple apps...")
results = verify_app_initialization(test_apps, env)


### Understanding Device Paths vs Local Paths

**Important:** The snapshot and app data paths are on the Android device, not on your local machine!

- **Device paths** (Android emulator): `/data/data/org.wikipedia`, `/data/data/android_world/snapshots/org.wikipedia`
- **Local paths** (your Linux machine): `/home/ligu/projects/android_world_plus/...`

You cannot `cd` into device paths from your local terminal. You must use ADB commands to access them.


In [ ]:
# Helper function to check device paths using ADB
def check_device_paths(app_name, env):
    """
    Check if snapshot and app data paths exist on the Android device.
    
    Args:
        app_name: App name (e.g., "wikipedia")
        env: AsyncEnv instance
    """
    from android_world.utils import app_snapshot
    from android_world.env import adb_utils
    
    # Get package name
    activity = adb_utils.get_adb_activity(app_name)
    if not activity:
        print(f"⚠ Could not find activity for '{app_name}'")
        return
    
    package_name = adb_utils.extract_package_name(activity)
    print(f"Package name: {package_name}\n")
    
    # App data path
    app_data_path = f"/data/data/{package_name}"
    print(f"Checking app data path:")
    print(f"  Path: {app_data_path}")
    app_data_exists = file_utils.check_directory_exists(app_data_path, env)
    print(f"  Exists: {'✓ Yes' if app_data_exists else '✗ No'}\n")
    
    # Snapshot path
    snapshot_path = f"/data/data/android_world/snapshots/{package_name}"
    print(f"Checking snapshot path:")
    print(f"  Path: {snapshot_path}")
    snapshot_exists = file_utils.check_directory_exists(snapshot_path, env)
    print(f"  Exists: {'✓ Yes' if snapshot_exists else '✗ No'}\n")
    
    # Also show how to check via ADB directly
    print("Alternative: Check via ADB shell command:")
    print(f"  adb shell ls -la {app_data_path}")
    print(f"  adb shell ls -la {snapshot_path}")
    
    return {
        'app_data_path': app_data_path,
        'app_data_exists': app_data_exists,
        'snapshot_path': snapshot_path,
        'snapshot_exists': snapshot_exists
    }

# Check paths for Wikipedia
print("=" * 80)
print("Checking Device Paths for Wikipedia App")
print("=" * 80)
results = check_device_paths("wikipedia", env)


### Using ADB Commands to Access Device Paths

Since these paths are on the Android device (not your local machine), you need to use ADB commands to access them.

**⚠️ IMPORTANT: Permission Denied Error**

The `/data/data/` directory requires **root access** on Android. Regular ADB commands will show:
```
ls: /data/data/org.wikipedia: Permission denied
```

**✅ RECOMMENDED: Use `adb root` (AndroidWorld Method)**

This is how AndroidWorld handles root access internally:

```bash
# Step 1: Switch ADB to root mode (one-time per session)
adb root

# Step 2: Now regular commands work with root access
adb shell ls -la /data/data/org.wikipedia
adb shell ls -la /data/data/android_world/snapshots/org.wikipedia
adb shell [ -d /data/data/android_world/snapshots/org.wikipedia ] && echo "Exists" || echo "Does not exist"
```

**⚠️ Why `su -c` Doesn't Work:**

Many Android emulators/devices don't support the `su -c` syntax. You may see errors like:
- `su: invalid uid/gid '-c'`
- `su: failed to exec -c: No such file or directory`
- `su root -c '...'` also fails with similar errors

**❌ These methods often fail:**
```bash
adb shell su -c 'ls -la /data/data/org.wikipedia'           # ❌ invalid uid/gid
adb shell su root -c 'ls -la /data/data/org.wikipedia'     # ❌ failed to exec -c
adb shell su - root -c 'ls -la /data/data/org.wikipedia'    # ❌ also fails
```

**✅ Use this instead:**
```bash
adb root  # Switch to root mode
adb shell ls -la /data/data/org.wikipedia  # Now works!
```

**💡 BEST: Use Python with AndroidWorld**

AndroidWorld automatically handles root access via `adb root`:
```python
file_utils.check_directory_exists(path, env)  # Automatically uses 'adb root' when needed
```


In [ ]:
# Demonstrate checking paths using ADB commands directly
import subprocess

def check_device_path_via_adb(path):
    """Check if a path exists on the device using ADB shell command."""
    try:
        result = subprocess.run(
            ["adb", "shell", f"[ -d {path} ] && echo 'Exists' || echo 'Does not exist'"],
            capture_output=True,
            text=True,
            timeout=10
        )
        return result.stdout.strip()
    except Exception as e:
        return f"Error: {e}"

# Check Wikipedia paths
wikipedia_snapshot_path = "/data/data/android_world/snapshots/org.wikipedia"
wikipedia_app_data_path = "/data/data/org.wikipedia"

print("Checking device paths via ADB commands:")
print(f"\n1. Snapshot path: {wikipedia_snapshot_path}")
print(f"   Status: {check_device_path_via_adb(wikipedia_snapshot_path)}")

print(f"\n2. App data path: {wikipedia_app_data_path}")
print(f"   Status: {check_device_path_via_adb(wikipedia_app_data_path)}")

print("\nNote: If you see permission errors, the device may need root access.")
print("AndroidWorld automatically handles root access via env.controller.")
